In [2]:
import json
from google.colab import files

uploaded = files.upload()  # upload ton train.json

filename = list(uploaded.keys())[0]
with open(filename, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Nombre d'exemples : {len(data)}")
print(f"Premier exemple :")
print(json.dumps(data[0], ensure_ascii=False, indent=2))

Saving train.json to train.json
Nombre d'exemples : 9500
Premier exemple :
{
  "input": "Lors de mon séjour à Nouakchott, comment demander si c'est loin en hassaniya ?",
  "output": "On dit 'baiid ?' ou 'hadha baiid ?' pour demander si c'est loin en hassaniya.",
  "categorie": "transport",
  "français": "C'est loin ?",
  "hassaniya": "baiid ? / hadha baiid ?"
}


In [3]:
# Construire le vocabulaire à partir du dataset
corpus = ""
for exemple in data:
    corpus += exemple["input"] + " " + exemple["output"] + " "

# Vocabulaire unique de caractères
chars = sorted(set(corpus))
vocab_size = len(chars)

# Mappings caractère ↔ entier
char2idx = {ch: i for i, ch in enumerate(chars)}
idx2char = {i: ch for i, ch in enumerate(chars)}

print(f"Taille du corpus : {len(corpus):,} caractères")
print(f"Taille du vocabulaire : {vocab_size} caractères uniques")
print(f"Extrait du vocabulaire : {chars[:20]}")

Taille du corpus : 2,098,522 caractères
Taille du vocabulaire : 92 caractères uniques
Extrait du vocabulaire : ['\n', ' ', '!', '#', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6']


In [4]:
import torch

# Encoder tout le corpus en entiers
def encode(text):
    return [char2idx[ch] for ch in text if ch in char2idx]

def decode(indices):
    return ''.join([idx2char[i] for i in indices])

# Créer les séquences d'entraînement
sequences = []
for exemple in data:
    texte = exemple["input"] + " → " + exemple["output"]
    encoded = encode(texte)
    sequences.append(encoded)

# Vérification
print(f"Nombre de séquences : {len(sequences)}")
print(f"Longueur moyenne : {sum(len(s) for s in sequences) // len(sequences)} caractères")
print(f"\nTest encode/decode :")
test = "sbah lkhayr"
print(f"Original  : {test}")
print(f"Encodé    : {encode(test)}")
print(f"Décodé    : {decode(encode(test))}")

Nombre de séquences : 9500
Longueur moyenne : 221 caractères

Test encode/decode :
Original  : sbah lkhayr
Encodé    : [69, 52, 51, 58, 1, 62, 61, 58, 51, 75, 68]
Décodé    : sbah lkhayr


In [5]:
from torch.utils.data import Dataset, DataLoader

SEQ_LEN = 128  # longueur de chaque séquence d'entraînement

class HassaniyaDataset(Dataset):
    def __init__(self, sequences, seq_len):
        self.seq_len = seq_len
        # Aplatir toutes les séquences en un seul vecteur
        self.data = []
        for seq in sequences:
            self.data.extend(seq)
        self.data = torch.tensor(self.data, dtype=torch.long)
        print(f"Dataset : {len(self.data):,} tokens au total")

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y

dataset_torch = HassaniyaDataset(sequences, SEQ_LEN)
loader = DataLoader(dataset_torch, batch_size=64, shuffle=True)

print(f"Nombre de batchs : {len(loader)}")

Dataset : 2,108,022 tokens au total
Nombre de batchs : 32936


In [12]:
import torch.nn as nn
import math

class GuppyLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, num_heads=8,
                 num_layers=6, seq_len=128, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(seq_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device)
        x = self.embed(x) + self.pos_embed(positions)

        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        x = self.transformer(x, mask=mask, is_causal=True)
        x = self.norm(x)
        return self.head(x)

device = torch.device("cuda")
model = GuppyLM(
    vocab_size=vocab_size,
    embed_dim=384,       # 256 → 384
    num_heads=8,
    num_layers=6,
    seq_len=SEQ_LEN,
    dropout=0.3
).to(device)

total = sum(p.numel() for p in model.parameters())
print(f"Paramètres : {total:,} (~{total//1_000_000}M)")


Paramètres : 10,767,452 (~10M)


In [9]:
import wandb

# Connexion W&B
wandb.login()  # il va te demander une clé API — copie-la depuis wandb.ai/settings

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 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


wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24066 (24066-esp) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [13]:
# Stopper le run précédent
wandb.finish()

# Nouveau run corrigé
wandb.init(
    project="sederni-hassaniya",
    name="guppylm-pretrain-v2",
    config={
        "vocab_size": vocab_size,
        "embed_dim": 384,
        "num_heads": 8,
        "num_layers": 6,
        "seq_len": SEQ_LEN,
        "batch_size": 64,
        "learning_rate": 3e-4,
        "epochs": 3,
        "max_steps_per_epoch": 500,  # on limite à 500 batchs par epoch
    }
)

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(3):
    total_loss = 0
    for batch_idx, (x, y) in enumerate(loader):

        # Stopper à 500 batchs par epoch
        if batch_idx >= 500:
            break

        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            avg_loss = total_loss / (batch_idx + 1)
            wandb.log({"loss": loss.item(), "avg_loss": avg_loss, "epoch": epoch})
            print(f"Epoch {epoch+1} | Batch {batch_idx}/500 | Loss : {loss.item():.4f}")

    print(f"\n✅ Epoch {epoch+1} | Loss moyenne : {total_loss/500:.4f}\n")

wandb.finish()
print("🎉 Entraînement terminé !")

avg_loss,▁█▇▇▇
epoch,▁▁▁▁▁
loss,▁█▄▃▁
avg_loss,0.10042
epoch,0
loss,0.09505


Epoch 1 | Batch 0/500 | Loss : 4.6284
Epoch 1 | Batch 100/500 | Loss : 2.0338
Epoch 1 | Batch 200/500 | Loss : 1.3687
Epoch 1 | Batch 300/500 | Loss : 1.0546
Epoch 1 | Batch 400/500 | Loss : 0.8442

✅ Epoch 1 | Loss moyenne : 1.4221

Epoch 2 | Batch 0/500 | Loss : 0.6915
Epoch 2 | Batch 100/500 | Loss : 0.6287
Epoch 2 | Batch 200/500 | Loss : 0.5025
Epoch 2 | Batch 300/500 | Loss : 0.4232
Epoch 2 | Batch 400/500 | Loss : 0.3878

✅ Epoch 2 | Loss moyenne : 0.4818

Epoch 3 | Batch 0/500 | Loss : 0.3148
Epoch 3 | Batch 100/500 | Loss : 0.3057
Epoch 3 | Batch 200/500 | Loss : 0.2722
Epoch 3 | Batch 300/500 | Loss : 0.2354
Epoch 3 | Batch 400/500 | Loss : 0.3232

✅ Epoch 3 | Loss moyenne : 0.2627



avg_loss,█▄▄▃▃▂▂▂▁▁▁▁▁▁▁
epoch,▁▁▁▁▁▅▅▅▅▅█████
loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
avg_loss,0.2722
epoch,2
loss,0.32323


🎉 Entraînement terminé !


In [14]:
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': vocab_size,
    'char2idx': char2idx,
    'idx2char': idx2char,
    'config': {
        'embed_dim': 384,
        'num_heads': 8,
        'num_layers': 6,
        'seq_len': SEQ_LEN,
    }
}, 'sederni_guppylm.pt')

print("✅ Modèle sauvegardé : sederni_guppylm.pt")

✅ Modèle sauvegardé : sederni_guppylm.pt


In [15]:
def generer(prompt, nb_caracteres=100):
    model.eval()
    encoded = encode(prompt)
    x = torch.tensor(encoded, dtype=torch.long).unsqueeze(0).to(device)

    resultat = prompt
    for _ in range(nb_caracteres):
        with torch.no_grad():
            logits = model(x[:, -SEQ_LEN:])
            probs = torch.softmax(logits[0, -1] / 0.8, dim=0)
            next_char = torch.multinomial(probs, 1).item()
        resultat += idx2char[next_char]
        x = torch.cat([x, torch.tensor([[next_char]]).to(device)], dim=1)

    return resultat

# Test avec des prompts Hassaniya
print(generer("sbah lkhayr →"))
print("---")
print(generer("bkem ? →"))
print("---")
print(generer("ndor thiéboudienne →"))

sbah lkhayr → In Hassaniya, you say 'sbah lkhayr'. The response is also 'sbah lkhayr' or 'sbah nour' (morning of 
---
bkem ? → On dit 'lhouman' pour dire je ne veux pas en hassaniya. À Nouakchott, les mois les plus chauds sont
---
ndor thiéboudienne → On dit 'ndor nedwiya' pour dire je viens de France, ou 'ane jay mn Europe' pour dire je viens d'Eur


In [16]:
# Télécharger le modèle sur ton PC
from google.colab import files
files.download('sederni_guppylm.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Analyse des résultats du modèle GuppyLM

### Observations

Lors des tests de génération, le modèle montre des résultats mitigés :

| Prompt | Résultat | Évaluation |
|--------|----------|------------|
| `sbah lkhayr →` | Réponse correcte sur les salutations |  Correct |
| `bkem ? →` | Confond avec `lhouman` (il fait chaud) |  Incorrect |
| `ndor thiéboudienne →` | Mélange deux réponses distinctes |  Incorrect |

### Causes identifiées

**1. Entraînement insuffisant**
Le modèle a été entraîné sur 500 batchs × 3 epochs = 1 500 steps,
soit environ 5% des 32 936 batchs disponibles. Le modèle n'a pas vu
assez de données pour généraliser.

**2. Architecture caractère par caractère**
GuppyLM prédit le prochain caractère de façon statistique.
Il n'a aucune compréhension sémantique — il ne sait pas que
`bkem` (combien ?) et `lhouman` (il fait chaud) appartiennent
à des catégories différentes.

**3. Redondance du dataset**
Les 9 714 paires générées par duplication sont trop similaires
aux 286 paires originales. Le modèle mélange les réponses
de paires structurellement proches.

### Pistes d'amélioration

- **Plus d'epochs** : entraîner sur l'ensemble du dataset (32 936 batchs)
- **Tokenisation subword (BPE)** : mieux adapté au hassaniya que
  le niveau caractère
- **Dataset moins redondant** : augmenter les seeds authentiques
  plutôt que de dupliquer
- **Architecture seq2seq** : encoder-décodeur plus adapté
  à la tâche question-réponse que le modèle de langue causal

### Conclusion

Les limites observées sont inhérentes à l'approche GuppyLM sur
une langue à faibles ressources avec un dataset synthétique.
Pour une application réelle, un modèle pré-entraîné multilingue
(comme mBERT ou AraBART) fine-tuné sur le hassaniya serait
plus approprié.